In [1]:
import os, json, time, glob, warnings
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

def find_base(root="/content/asvspoof"):
    if not os.path.exists(root):
        return None
    for dp, dn, fn in os.walk(root):
        if "ASVspoof2019_LA_train" in dn:
            return dp
    return None

base = find_base()
print("voice_features.npz on Drive:", os.path.exists("/content/drive/MyDrive/voice_features.npz"))
print("ASVspoof audio folder:", base)

Mounted at /content/drive
GPU: Tesla T4
voice_features.npz on Drive: True
ASVspoof audio folder: None


In [3]:
print("base =", base)
print("asvspoof folder exists:", os.path.exists("/content/asvspoof"))
print("LA.zip exists:", os.path.exists("/content/asvspoof/LA.zip"))

base = None
asvspoof folder exists: False
LA.zip exists: False


In [4]:
import subprocess, time, requests, hashlib, signal
subprocess.run("apt-get install -y -qq aria2 > /dev/null 2>&1", shell=True)

data = requests.get("https://zenodo.org/api/records/6906306").json()
la_file = next(f for f in data["files"] if f["key"] == "LA.zip")
download_url = la_file["links"]["self"]
expected_bytes = la_file["size"]
expected_md5 = (la_file.get("checksum") or "").replace("md5:", "")
print("File:", la_file["key"], "| size:", round(expected_bytes / 1024**3, 2), "GB")

File: LA.zip | size: 7.12 GB


In [5]:
os.makedirs("/content/asvspoof", exist_ok=True)
zip_path = "/content/asvspoof/LA.zip"
log_path = "/content/asvspoof/download.log"

running = subprocess.run(["pgrep", "-x", "aria2c"], capture_output=True, text=True).stdout.split() + \
          subprocess.run(["pgrep", "-x", "wget"], capture_output=True, text=True).stdout.split()
if running:
    print("A downloader is already running (pids:", running, "). Do NOT start another.")
else:
    cmd = ["aria2c", "-x", "4", "-s", "4", "-k", "20M", "-c", "--file-allocation=none",
           "--max-tries=20", "--retry-wait=5", "--console-log-level=warn",
           "-d", "/content/asvspoof", "-o", "LA.zip", download_url]
    proc = subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    print("Download started in the background. pid =", proc.pid)

Download started in the background. pid = 3926


In [27]:
def written_mb():
    if not os.path.exists(zip_path): return 0.0
    return os.stat(zip_path).st_blocks * 512 / 1024**2

total_mb = expected_bytes / 1024**2
a = written_mb(); time.sleep(20); b = written_mb()
speed = (b - a) / 20
print(f"{b:,.0f} / {total_mb:,.0f} MB  ({b / total_mb * 100:.1f}%)")
print(f"speed: {speed:.1f} MB/s", end="")
print(f" | about {(total_mb - b) / speed / 60:.0f} min left" if speed > 0.05 else " | not moving")
print("download process:", "running" if proc.poll() is None else f"finished (exit code {proc.returncode})")

7,287 / 7,287 MB  (100.0%)
speed: 0.0 MB/s | not moving
download process: finished (exit code 0)


In [28]:
assert proc.poll() is not None and proc.returncode == 0, "Not finished yet. Wait and re-run Step 3."
actual = os.path.getsize(zip_path)
print("Size matches Zenodo:", actual == expected_bytes, f"({actual:,} vs {expected_bytes:,})")
h = hashlib.md5()
with open(zip_path, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
print("md5 matches Zenodo:", h.hexdigest() == expected_md5)

Size matches Zenodo: True (7,640,952,520 vs 7,640,952,520)
md5 matches Zenodo: True


In [30]:
assert os.path.getsize(zip_path) == expected_bytes, "Zip is incomplete. Go back to Step 4."
if find_base() is None:
    subprocess.run(["unzip", "-q", "-o", zip_path, "-d", "/content/asvspoof/"], check=True)
base = find_base()
print("Base folder:", base)
print(sorted(os.listdir(base)))

Base folder: /content/asvspoof/LA
['ASVspoof2019_LA_asv_protocols', 'ASVspoof2019_LA_asv_scores', 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019_LA_dev', 'ASVspoof2019_LA_eval', 'ASVspoof2019_LA_train', 'README.LA.txt']


In [31]:
d = np.load("/content/drive/MyDrive/voice_features.npz")
names = {k: d[f"names_{k}"] for k in ["train", "val", "test"]}
atk   = {k: d[f"atk_{k}"]   for k in ["train", "val", "test"]}
y     = {k: d[f"y_{k}"]     for k in ["train", "val", "test"]}
dirs = {"train": os.path.join(base, "ASVspoof2019_LA_train", "flac"),
        "val":   os.path.join(base, "ASVspoof2019_LA_dev",   "flac"),
        "test":  os.path.join(base, "ASVspoof2019_LA_eval",  "flac")}
for k in dirs:
    ok = os.path.exists(os.path.join(dirs[k], names[k][0] + ".flac"))
    print(f"{k}: {len(names[k])} clips | real {(y[k]==0).sum()} | fake {(y[k]==1).sum()} | audio found: {ok}")

train: 12780 clips | real 2580 | fake 10200 | audio found: True
val: 7348 clips | real 2548 | fake 4800 | audio found: True
test: 3998 clips | real 800 | fake 3198 | audio found: True


In [32]:
import librosa
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model

SR, TARGET = 16000, 64000          # 4 seconds
USE_FP16 = True

def load_clip(path):
    y_, _ = librosa.load(path, sr=SR, duration=8.0)
    y_, _ = librosa.effects.trim(y_, top_db=30)               # remove edge silence
    if len(y_) < 400:
        y_ = np.zeros(TARGET, dtype=np.float32) + 1e-4
    if len(y_) < TARGET:
        y_ = np.tile(y_, int(np.ceil(TARGET / len(y_))))      # repeat-pad, no artificial silence
    y_ = y_[:TARGET].astype(np.float32)
    return (y_ - y_.mean()) / np.sqrt(y_.var() + 1e-7)        # per-clip normalization

class ClipSet(Dataset):
    def __init__(self, nm, folder): self.nm, self.folder = nm, folder
    def __len__(self): return len(self.nm)
    def __getitem__(self, i):
        return torch.from_numpy(load_clip(os.path.join(self.folder, self.nm[i] + ".flac")))

MODEL_NAME = "facebook/wav2vec2-xls-r-300m"
w2v = Wav2Vec2Model.from_pretrained(MODEL_NAME).to(device).eval()
n_layers = w2v.config.num_hidden_layers + 1
print("Layers (incl. input embedding):", n_layers, "| hidden size:", w2v.config.hidden_size)

xb = next(iter(DataLoader(ClipSet(names["train"][:3], dirs["train"]), batch_size=3)))
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
    hs = w2v(xb.to(device), output_hidden_states=True).hidden_states
pooled = torch.stack([h.float().mean(dim=1) for h in hs], dim=1)
print("clips:", tuple(xb.shape), "| layers:", len(hs), "| frames per clip:", hs[0].shape[1],
      "| pooled:", tuple(pooled.shape), "| finite:", bool(torch.isfinite(pooled).all()))

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.27GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Layers (incl. input embedding): 25 | hidden size: 1024


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.27GB            

model.safetensors: downloading bytes:           |  0.00B            

clips: (3, 64000) | layers: 25 | frames per clip: 199 | pooled: (3, 25, 1024) | finite: True


In [33]:
tag = MODEL_NAME.split("/")[-1]
feat_path = f"/content/drive/MyDrive/voice_ssl_{tag}.npz"

def extract(split):
    dl = DataLoader(ClipSet(names[split], dirs[split]), batch_size=32, num_workers=2)
    out, t0 = [], time.time()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
        for i, xb in enumerate(dl):
            hs = w2v(xb.to(device), output_hidden_states=True).hidden_states
            pooled = torch.stack([h.float().mean(dim=1) for h in hs], dim=1)
            out.append(pooled.cpu().numpy().astype(np.float16))
            if i % 50 == 0:
                print(f"  {split}: batch {i}/{len(dl)} | {time.time()-t0:.0f}s")
    return np.concatenate(out)

if os.path.exists(feat_path):
    dd = np.load(feat_path)
    F = {k: dd[k] for k in ["train", "val", "test"]}
    print("Loaded saved features from Drive.")
else:
    F = {}
    for split in ["train", "val", "test"]:
        print(f"Extracting {split}...")
        F[split] = extract(split)
        print(f"  done: {F[split].shape} | all finite: {bool(np.isfinite(F[split].astype(np.float32)).all())}")
    np.savez(feat_path, **F)
    print("Saved to Drive:", feat_path)
for k in F: print(k, F[k].shape)

Extracting train...
  train: batch 0/400 | 1s
  train: batch 50/400 | 22s
  train: batch 100/400 | 44s
  train: batch 150/400 | 66s
  train: batch 200/400 | 88s
  train: batch 250/400 | 109s
  train: batch 300/400 | 131s
  train: batch 350/400 | 154s
  done: (12780, 25, 1024) | all finite: True
Extracting val...
  val: batch 0/230 | 1s
  val: batch 50/230 | 22s
  val: batch 100/230 | 44s
  val: batch 150/230 | 66s
  val: batch 200/230 | 88s
  done: (7348, 25, 1024) | all finite: True
Extracting test...
  test: batch 0/125 | 1s
  test: batch 50/125 | 23s
  test: batch 100/125 | 44s
  done: (3998, 25, 1024) | all finite: True
Saved to Drive: /content/drive/MyDrive/voice_ssl_wav2vec2-xls-r-300m.npz
train (12780, 25, 1024)
val (7348, 25, 1024)
test (3998, 25, 1024)


In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score, balanced_accuracy_score, confusion_matrix
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

def compute_eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return (fpr[i] + fnr[i]) / 2

HOLD = ["A05", "A06"]                                   # two attack methods hidden from training in this step
tr_m = ~np.isin(atk["train"], HOLD)                     # train on real + A01-A04
va_m = (atk["val"] == "-") | np.isin(atk["val"], HOLD)  # check on real + A05, A06
print(f"Selection split -> train {tr_m.sum()} | check {va_m.sum()} | held-out attacks: {HOLD}")

def fit_eval(layer, C):
    Xa = F["train"][tr_m, layer].astype(np.float32); ya = y["train"][tr_m]
    Xb = F["val"][va_m, layer].astype(np.float32);   yb = y["val"][va_m]
    sc = StandardScaler().fit(Xa)
    clf = LogisticRegression(C=C, class_weight="balanced", max_iter=300).fit(sc.transform(Xa), ya)
    return compute_eer(yb, clf.predict_proba(sc.transform(Xb))[:, 1])

layer_eer = []
for layer in range(n_layers):
    e = fit_eval(layer, 0.1)
    layer_eer.append(e)
    print(f"layer {layer:2d}: EER on held-out attacks {e*100:5.2f}%")
best_layer = int(np.argmin(layer_eer))
print(f"\nBest layer: {best_layer} ({layer_eer[best_layer]*100:.2f}%)")

c_eer = {C: fit_eval(best_layer, C) for C in [0.001, 0.01, 0.1, 1.0]}
best_C = min(c_eer, key=c_eer.get)
print("Regularization C scan:", {C: round(e*100, 2) for C, e in c_eer.items()}, "-> best C:", best_C)

Selection split -> train 9380 | check 4148 | held-out attacks: ['A05', 'A06']
layer  0: EER on held-out attacks  0.68%
layer  1: EER on held-out attacks  0.31%
layer  2: EER on held-out attacks  0.09%
layer  3: EER on held-out attacks  0.16%
layer  4: EER on held-out attacks  0.07%
layer  5: EER on held-out attacks  0.49%
layer  6: EER on held-out attacks  1.06%
layer  7: EER on held-out attacks  1.25%
layer  8: EER on held-out attacks  1.97%
layer  9: EER on held-out attacks  2.20%
layer 10: EER on held-out attacks  3.29%
layer 11: EER on held-out attacks  3.86%
layer 12: EER on held-out attacks  4.79%
layer 13: EER on held-out attacks  5.87%
layer 14: EER on held-out attacks  5.13%
layer 15: EER on held-out attacks  4.85%
layer 16: EER on held-out attacks  4.17%
layer 17: EER on held-out attacks  4.14%
layer 18: EER on held-out attacks  3.81%
layer 19: EER on held-out attacks  2.26%
layer 20: EER on held-out attacks  2.36%
layer 21: EER on held-out attacks  2.46%
layer 22: EER on hel

In [35]:
Xtr = F["train"][:, best_layer].astype(np.float32)
scaler = StandardScaler().fit(Xtr)
clf = LogisticRegression(C=best_C, class_weight="balanced", max_iter=1000).fit(scaler.transform(Xtr), y["train"])

def probs_for(split):
    return clf.predict_proba(scaler.transform(F[split][:, best_layer].astype(np.float32)))[:, 1]

def summarize(probs, labels, title):
    preds = (probs >= 0.5).astype(int)
    print(title)
    print(f"  EER: {compute_eer(labels, probs)*100:.2f}%   (lower is better, 50% = guessing)")
    print(f"  Balanced accuracy: {balanced_accuracy_score(labels, preds)*100:.2f}%")
    print(f"  Real correct: {(preds[labels==0]==0).mean()*100:.2f}% | Fake correct: {(preds[labels==1]==1).mean()*100:.2f}%")
    print(f"  AUC: {roc_auc_score(labels, probs):.4f}")
    print(f"  Confusion matrix:\n{confusion_matrix(labels, preds)}")

val_probs = probs_for("val")
summarize(val_probs, y["val"], "DEV (same attack types as training)")

# ---- TEST: run once ----
test_probs = probs_for("test")
summarize(test_probs, y["test"], "TEST (eval partition, mostly UNSEEN attack types)")
print("\nPrevious CNN (v1) on the same test clips: EER 21.44% | AUC 0.8586 | balanced acc 78.96%")

test_df = pd.DataFrame({"file": names["test"], "attack": atk["test"], "label": y["test"], "spoof_prob": test_probs})
print("\nFakes detected per attack type (threshold 0.5):")
print(test_df[test_df.label == 1].groupby("attack")["spoof_prob"].agg(n="count", detected=lambda s: (s >= 0.5).mean() * 100).round(1))

DEV (same attack types as training)
  EER: 0.16%   (lower is better, 50% = guessing)
  Balanced accuracy: 99.85%
  Real correct: 99.80% | Fake correct: 99.90%
  AUC: 1.0000
  Confusion matrix:
[[2543    5]
 [   5 4795]]
TEST (eval partition, mostly UNSEEN attack types)
  EER: 6.50%   (lower is better, 50% = guessing)
  Balanced accuracy: 87.65%
  Real correct: 99.62% | Fake correct: 75.67%
  AUC: 0.9837
  Confusion matrix:
[[ 797    3]
 [ 778 2420]]

Previous CNN (v1) on the same test clips: EER 21.44% | AUC 0.8586 | balanced acc 78.96%

Fakes detected per attack type (threshold 0.5):
          n  detected
attack               
A07     246      99.6
A08     246      98.0
A09     246      96.7
A10     246       8.1
A11     246      41.5
A12     246      38.6
A13     246      99.2
A14     246      87.0
A15     246      18.7
A16     246      98.8
A17     246      98.0
A18     246      99.6
A19     246     100.0


In [36]:
import pickle, sklearn, transformers
with open("/content/drive/MyDrive/voice_model_v2.pkl", "wb") as f:
    pickle.dump({"scaler": scaler, "clf": clf, "layer": best_layer, "ssl_model": MODEL_NAME}, f)

pd.DataFrame({"file": names["val"], "label": y["val"], "spoof_prob": val_probs}).to_csv("/content/drive/MyDrive/voice_v2_val_scores.csv", index=False)
test_df.to_csv("/content/drive/MyDrive/voice_v2_test_scores.csv", index=False)

info = {
    "model": f"Frozen {MODEL_NAME}; mean-pooled hidden states of one layer; StandardScaler + LogisticRegression",
    "version": "v2 (replaces CNN v1 as the voice module if it scores better)",
    "layer": best_layer, "C": best_C,
    "layer_selection": {"protocol": "train on real + A01-A04, check on real + A05-A06 (held-out attacks)",
                        "eer_per_layer": [round(float(e), 4) for e in layer_eer]},
    "preprocessing": {"sr": SR, "trim_top_db": 30, "length_samples": TARGET,
                      "pad": "repeat", "normalize": "per-clip zero mean, unit variance"},
    "data": "same subsets as v1 (ASVspoof2019 LA; 12780 train / 7348 dev / 3998 eval-subset)",
    "class_weight": "balanced", "seed": "n/a (deterministic given features)",
    "versions": {"sklearn": sklearn.__version__, "transformers": transformers.__version__, "torch": torch.__version__},
    "dev_eer": float(compute_eer(y["val"], val_probs)), "dev_auc": float(roc_auc_score(y["val"], val_probs)),
    "test_eer": float(compute_eer(y["test"], test_probs)), "test_auc": float(roc_auc_score(y["test"], test_probs)),
    "test_balanced_acc": float(balanced_accuracy_score(y["test"], (test_probs >= 0.5).astype(int))),
    "v1_cnn_test": {"eer": 0.2144, "auc": 0.8586, "balanced_acc": 0.7896},
    "limitation": "Trained and tested on clean studio speech (ASVspoof 2019 LA); phone-call/compressed audio not evaluated.",
}
with open("/content/drive/MyDrive/voice_model_v2_info.json", "w") as f:
    json.dump(info, f, indent=2)
print("Saved. Test EER:", round(info["test_eer"]*100, 2), "% | AUC:", round(info["test_auc"], 4))

Saved. Test EER: 6.5 % | AUC: 0.9837
